In [1]:
from elos.elo_tracker import EloTracker
from utils.utils import load_all_games_csv, get_teams, plot_elo_ratings_over_time
from simulation.simulation import simulate_seasons
import numpy as np
import pandas as pd

# Simulation

This notebook focuses on simulating the 2025 season using Elo ratings, estimating each team's probability of making it to the playoffs and different rounds within them.

## Get Divisions and Leagues for Past Season Teams

In [2]:
# Get divisions and leagues for each team
AL_EAST = {'TOR', 'NYA', 'BOS', 'TBA', 'BAL'}
AL_CENTRAL = {'CLE', 'DET', 'KCA', 'MIN', 'CHA'}
AL_WEST = {'SEA', 'HOU', 'TEX', 'ATH', 'ANA'}

AL = [AL_EAST, AL_CENTRAL, AL_WEST]

NL_EAST = {'PHI', 'NYN', 'MIA', 'ATL', 'WAS'}
NL_CENTRAL = {'MIL', 'CHN', 'CIN', 'SLN', 'PIT'}
NL_WEST = {'LAN', 'SDN', 'SFN', 'ARI', 'COL'}

NL = NL_EAST, NL_CENTRAL, NL_WEST

CURRENT_TEAMS = AL_EAST | AL_CENTRAL | AL_WEST | NL_EAST | NL_CENTRAL | NL_WEST

In [3]:
PAST_SEASON_TEAMS = CURRENT_TEAMS.copy()
# A's moved
PAST_SEASON_TEAMS.discard('ATH')
PAST_SEASON_TEAMS.add('OAK')

## Get Elo Ratings for each team

In [4]:
all_games = load_all_games_csv('../data/gameinfo_cleaned.csv')

teams = get_teams(all_games)
et = EloTracker(teams)

et.add_history(all_games)

# Get dict of latest elos for past teams
elos_map = {team: et.elos_map[team][-1][3] for team in PAST_SEASON_TEAMS}

# Revert to mean by 1/3 for new season
elos_map = {team: elos_map[team] + (1500 - elos_map[team]) / 3 for team in PAST_SEASON_TEAMS}

# Change OAK to ATH for A's move
elos_map['ATH'] = elos_map['OAK']
del elos_map['OAK']

elos_map

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/utils.py:27: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


{'PHI': 1522.731826710146,
 'ARI': 1514.5682422324635,
 'DET': 1505.9052275046763,
 'TOR': 1495.5159210103802,
 'COL': 1462.099952940796,
 'CIN': 1491.346298436719,
 'SEA': 1510.3629520246325,
 'ANA': 1464.2556058241114,
 'NYA': 1521.865698858941,
 'MIA': 1474.8425358668817,
 'ATL': 1522.6776363652439,
 'KCA': 1491.9308675960867,
 'SLN': 1501.475780378102,
 'PIT': 1486.1727607963771,
 'CHN': 1502.8983221771189,
 'HOU': 1520.0027065050556,
 'CHA': 1432.28608563123,
 'BOS': 1498.0906207045605,
 'MIN': 1498.029487442219,
 'CLE': 1510.3684505749704,
 'BAL': 1517.1047622196943,
 'TBA': 1509.6581249129413,
 'TEX': 1498.5358956818375,
 'WAS': 1476.3742822615345,
 'NYN': 1516.1993638221697,
 'MIL': 1520.8962222440791,
 'SDN': 1523.9338488249748,
 'SFN': 1498.8203286993905,
 'LAN': 1543.4531570264198,
 'ATH': 1467.6783484973278}

## Get Schedule

In [5]:
schedule_df = pd.read_csv('../data/2025schedule.csv')
schedule_df.head()

,Date,Num,Day,Visitor,League,Game,Home,League.1,Game.1,Day/Night,Location,Postponed,Makeup
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN


In [6]:
# Get as list
schedule = [(home, away) for home, away in zip(schedule_df['Home'], schedule_df['Visitor'])]

## Simulate Seasons

In [13]:
team_results = simulate_seasons(10000, schedule, AL, NL, elos_map)

100%|██████████| 10000/10000 [01:16<00:00, 130.14it/s]


In [14]:
# Get DF of results
avg_wins = [team_results[team][0] for team in CURRENT_TEAMS]
playoff_pcts = [team_results[team][1] for team in CURRENT_TEAMS]
div_pcts = [team_results[team][2] for team in CURRENT_TEAMS]
champ_pcts = [team_results[team][3] for team in CURRENT_TEAMS]
ws_pcts = [team_results[team][4] for team in CURRENT_TEAMS]
winner_pcts = [team_results[team][5] for team in CURRENT_TEAMS]

# Make PD dataframe to easily visualize
results_df = pd.DataFrame({'Team':list(CURRENT_TEAMS), 'Avg. Wins':avg_wins, 'Playoff %':playoff_pcts,
                           'Divisional %':div_pcts, 'Championship %':champ_pcts, 'WS %':ws_pcts, 'Win WS %':winner_pcts})
results_df = results_df.sort_values(by='Win WS %', ascending=False).reset_index(drop=True)
results_df

,Team,Avg. Wins,Playoff %,Divisional %,Championship %,WS %,Win WS %
0,LAN,91.4759,79.55,55.47,31.21,17.47,10.51
1,NYA,86.3577,61.02,44.09,23.38,12.74,6.23
2,MIL,85.9030,58.91,44.85,22.96,11.25,6.05
3,PHI,86.2702,59.47,41.11,21.16,10.93,5.97
4,ATL,86.2514,61.26,43.39,22.17,11.00,5.75
5,HOU,86.8153,66.51,41.77,21.71,11.50,5.53
6,BAL,85.2694,58.09,41.31,22.06,11.61,5.53
7,CLE,84.4571,57.31,43.98,22.16,10.64,5.14
8,SDN,86.3098,57.22,34.63,17.56,9.36,4.84
9,NYN,84.4814,50.66,34.39,17.47,8.56,4.45
